<a href="https://colab.research.google.com/github/srihanreddy/Reinforcement-Learning/blob/main/Lab_Assignment_12_Value_Iteration_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: Value Iteration in a Deterministic Grid World

In [1]:
import pandas as pd
import numpy as np

# 1. Define MDP Parameters
states = ['S0', 'S1', 'G']
non_terminal_states = ['S0', 'S1']
actions = ['Up', 'Down', 'Left', 'Right']
gamma = 0.9
theta = 1e-6

# Transition and reward rules based on the prompt's deterministic environment:
# S0:
# - Down -> S1 (reward: -1)
# - Up/Left/Right -> Stay S0 (reward: -1)
# S1:
# - Right -> G (reward: +100 as specified in 'each action has one next state. Use reward +100 when next is G')
# - Up -> S0 (reward: -1)
# - Down/Left -> Stay S1 (reward: -1)

def transition_and_reward(state, action):
    if state == 'S0':
        if action == 'Down':
            return 'S1', -1.0
        else:
            return 'S0', -1.0
    elif state == 'S1':
        if action == 'Right':
            return 'G', 100.0
        elif action == 'Up':
            return 'S0', -1.0
        else:
            return 'S1', -1.0
    return 'G', 0.0

# Print MDP Details
print("--- MDP Configuration ---")
for s in non_terminal_states:
    for a in actions:
        next_s, r = transition_and_reward(s, a)
        print(f"State: {s} | Action: {a:5} | Next State: {next_s:2} | Reward: {r:4.1f}")

--- MDP Configuration ---
State: S0 | Action: Up    | Next State: S0 | Reward: -1.0
State: S0 | Action: Down  | Next State: S1 | Reward: -1.0
State: S0 | Action: Left  | Next State: S0 | Reward: -1.0
State: S0 | Action: Right | Next State: S0 | Reward: -1.0
State: S1 | Action: Up    | Next State: S0 | Reward: -1.0
State: S1 | Action: Down  | Next State: S1 | Reward: -1.0
State: S1 | Action: Left  | Next State: S1 | Reward: -1.0
State: S1 | Action: Right | Next State: G  | Reward: 100.0


In [2]:
# 2. Value Iteration Algorithm
V = {'S0': 0.0, 'S1': 0.0, 'G': 0.0}
iterations_data = []

iteration = 0
iterations_data.append({'Iteration': iteration, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'V(G)': V['G']})

while True:
    iteration += 1
    delta = 0
    V_new = V.copy()

    for s in non_terminal_states:
        q_values = []
        for a in actions:
            next_s, r = transition_and_reward(s, a)
            q_val = r + gamma * V[next_s]
            q_values.append(q_val)

        new_val = max(q_values)
        delta = max(delta, abs(new_val - V[s]))
        V_new[s] = new_val

    V = V_new.copy()
    iterations_data.append({'Iteration': iteration, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'V(G)': V['G']})

    if delta < theta:
        break

# Display iteration-wise state values
df_iterations = pd.DataFrame(iterations_data)
print("\n--- Iteration-wise State-Value Table ---")
display(df_iterations)


--- Iteration-wise State-Value Table ---


,Iteration,V(S0),V(S1),V(G)
0,0,0.0,0.0,0.0
1,1,-1.0,100.0,0.0
2,2,89.0,100.0,0.0
3,3,89.0,100.0,0.0


In [4]:
# 3. Compute Final Action-Value (Q-values) and Optimal Policy
optimal_policy = {}
q_table = []

for s in non_terminal_states:
    best_action = None
    best_q = -float('inf')
    for a in actions:
        next_s, r = transition_and_reward(s, a)
        q_val = r + gamma * V[next_s]
        q_table.append({'State': s, 'Action': a, 'Next State': next_s, 'Q*(s,a)': round(q_val, 2)})
        if q_val > best_q:
            best_q = q_val
            best_action = a
    optimal_policy[s] = (best_action, round(best_q, 2))

# Display Q-Value Table
df_q = pd.DataFrame(q_table)
print("\n--- Final Action-Value Table Q*(s, a) ---")
display(df_q)

# Display Optimal Policy & Final Values
print("\n--- Optimal State Values and Optimal Policy ---")
for s in non_terminal_states:
    act, val = optimal_policy[s]
    print(f"V*({s}) = {val} | Optimal Action: {act}")
print(f"V*(G)  = {V['G']} (Terminal)")


--- Final Action-Value Table Q*(s, a) ---


,State,Action,Next State,"Q*(s,a)"
0,S0,Up,S0,79.1
1,S0,Down,S1,89.0
2,S0,Left,S0,79.1
3,S0,Right,S0,79.1
4,S1,Up,S0,79.1
5,S1,Down,S1,89.0
6,S1,Left,S1,89.0
7,S1,Right,G,100.0



--- Optimal State Values and Optimal Policy ---
V*(S0) = 89.0 | Optimal Action: Down
V*(S1) = 100.0 | Optimal Action: Right
V*(G)  = 0.0 (Terminal)
